# Praia ou Montanha — estatística em Python

Notebook que acompanha a apresentação **Praia ou Montanha**: os mesmos cálculos e gráficos, feitos com **NumPy**, **SciPy**, **pandas** e **Matplotlib**.

**Como usar no Colab:** menu *Ambiente de execução → Executar tudo*. No final há um **painel interativo** para mudar confiança, margem, proporção e população ao vivo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display
from types import SimpleNamespace

NOME_A, NOME_B = 'Praia', 'Montanha'
COR_A, COR_B = '#1f8fc4', '#9a6332'
COR_ERRO, COR_CONF, COR_TINTA, COR_OK = '#d94848', '#9fd0ee', '#10293f', '#1f9d6b'

plt.rcParams.update({
    'figure.figsize': (10, 4.8), 'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 11,
    'axes.titleweight': 'bold', 'axes.titlesize': 13,
})

# Formatação no padrão brasileiro: 1.068 e 3,05
def fmt(x, d=0):
    return f'{x:,.{d}f}'.replace(',', '_').replace('.', ',').replace('_', '.')

def pct(x, d=1):
    return fmt(100 * x, d) + '%'

def sgn(x, d=1):
    return ('+' if x > 0 else '−' if x < 0 else '') + fmt(abs(x), d)

## 1. Parâmetros e cálculos

$$n_0=\frac{z^2\,p(1-p)}{\varepsilon^2}\qquad n=\frac{n_0}{1+\frac{n_0-1}{N}}\qquad EP=\sqrt{\frac{\hat p(1-\hat p)}{n}}\qquad \text{margem}=z\cdot EP$$

A função `calcular` concentra toda a matemática; os gráficos só leem o resultado.

In [ ]:
def calcular(N=15_000_000, conf=95.0, eps=3.0, pB=51.0, n=None):
    '''Reproduz os cálculos da apresentação. Se n for None ou 0, usa o n mínimo planejado.'''
    alpha = 1 - conf / 100
    z = stats.norm.ppf(1 - alpha / 2)              # valor crítico z_(α/2)
    e = eps / 100
    n0 = z**2 * 0.25 / e**2                         # pior caso: p = 0,5
    n_fin = n0 / (1 + (n0 - 1) / N)                 # correção para população finita
    n_req = min(N, int(np.ceil(n_fin - 1e-9)))
    n_plan = min(N, int(np.ceil(n0 - 1e-9)))
    n = max(2, min(N, int(n))) if n else n_plan

    p = pB / 100                                    # p̂: proporção observada para Montanha
    kB = int(np.floor(n * p + 0.5)); kA = n - kB
    s2 = max(p * (1 - p), 1e-9)
    ep = np.sqrt(s2 / n)                            # erro padrão
    moe = z * ep                                    # margem de erro real

    D = 2 * p - 1                                   # diferença Montanha − Praia
    ep_D = 2 * ep
    moe_D = z * ep_D
    ic_D = (D - moe_D, D + moe_D)
    empate = ic_D[0] <= 0 <= ic_D[1]
    lider = NOME_B if D > 0 else NOME_A if D < 0 else None
    n_poder = np.ceil((z + stats.norm.ppf(0.80))**2 * 4 * s2 / D**2) if D else np.inf
    veredito = 'Empate exato' if lider is None else ('Empate técnico' if empate else f'{lider} lidera')

    return SimpleNamespace(N=N, conf=conf / 100, alpha=alpha, z=z, e=e, n0=n0, n_req=n_req, n_plan=n_plan,
                           n=n, p=p, kA=kA, kB=kB, s2=s2, ep=ep, moe=moe, D=D, ep_D=ep_D, moe_D=moe_D,
                           ic_D=ic_D, empate=empate, lider=lider, n_poder=n_poder, veredito=veredito)

def resumo(R):
    dados = {
        'População N': fmt(R.N),
        'Confiança': f'{pct(R.conf)}  (z ≈ {fmt(R.z, 3)})',
        'Margem planejada': f'±{fmt(R.e * 100, 1)} p.p.',
        'Amostra mínima n₀': fmt(np.ceil(R.n0 - 1e-9)),
        'Com correção finita': fmt(R.n_req),
        'Amostra usada n': fmt(R.n),
        f'Votos {NOME_A} × {NOME_B}': f'{fmt(R.kA)} × {fmt(R.kB)}',
        'Erro padrão': f'{fmt(R.ep * 100, 2)} p.p.',
        'Margem real': f'±{fmt(R.moe * 100, 1)} p.p.',
        f'Diferença ({NOME_B} − {NOME_A})': f'{sgn(R.D * 100)} p.p.   IC = [{sgn(R.ic_D[0] * 100)}; {sgn(R.ic_D[1] * 100)}] p.p.',
        'Veredito': R.veredito,
        'n p/ desempatar (poder 80%)': fmt(R.n_poder) if np.isfinite(R.n_poder) else '∞',
    }
    display(pd.DataFrame.from_dict(dados, orient='index', columns=['Valor']))

R = calcular()
resumo(R)

## 2. Uma parte fala do todo — sorteando um “tubinho”

400 pessoas, sorteamos 40. Rode a célula de novo: p̂ muda a cada sorteio, mas gira em torno do p real.

In [ ]:
def g_populacao(R, semente=None):
    rng = np.random.default_rng(semente)
    pop = np.zeros(400, dtype=int)
    pop[:int(round(400 * R.p))] = 1
    rng.shuffle(pop)
    sel = np.zeros(400, dtype=bool)
    sel[rng.choice(400, size=40, replace=False)] = True
    xs, ys = np.arange(400) % 20, np.arange(400) // 20
    cores = np.where(pop == 1, COR_B, COR_A)
    fig, ax = plt.subplots(figsize=(5.8, 6))
    ax.scatter(xs[~sel], ys[~sel], c=cores[~sel], s=60, alpha=0.25)
    ax.scatter(xs[sel], ys[sel], c=cores[sel], s=120, edgecolors=COR_TINTA, linewidths=2)
    ax.set(xticks=[], yticks=[], aspect='equal',
           title=f'p real = {pct(pop.mean())}   ·   p̂ da amostra = {pct(pop[sel].mean())}')
    ax.invert_yaxis(); ax.grid(False)
    for s in ax.spines.values(): s.set_visible(False)

g_populacao(R); plt.show()

## 3. Os pilares do tamanho da amostra

**Variabilidade:** p(1−p) é máximo em p = 0,5 — por isso planejar com 0,5 é o cenário mais seguro.

In [ ]:
def g_variabilidade(R, ax=None):
    ax = ax or plt.subplots(figsize=(8, 4))[1]
    p = np.linspace(0, 1, 401)
    ax.fill_between(p, p * (1 - p), color=COR_A, alpha=0.15)
    ax.plot(p, p * (1 - p), color=COR_A, lw=2.5)
    ax.plot(0.5, 0.25, 'o', color=COR_ERRO, ms=8)
    ax.annotate('máximo 0,25', (0.5, 0.25), xytext=(0, 10), textcoords='offset points', ha='center', fontweight='bold')
    ax.plot(R.p, R.s2, 'o', color=COR_B, ms=9, mec='white', mew=2, label=f'p̂ = {fmt(R.p, 2)}')
    ax.set(xlabel='p', ylabel='p(1 − p)', ylim=(0, 0.29), title='Por que p = 0,5 é o pior caso?')
    ax.legend(frameon=False)
    return ax

g_variabilidade(R); plt.show()

**Confiança:** o valor z deixa a área de confiança no centro da Normal padrão e α/2 em cada cauda.

In [ ]:
def g_normal(R, ax=None):
    ax = ax or plt.subplots()[1]
    z = R.z
    lim = max(4, np.ceil(z + 0.8))
    x = np.linspace(-lim, lim, 800)
    y = stats.norm.pdf(x)
    centro = np.abs(x) <= z
    ax.fill_between(x, y, where=centro, interpolate=True, color=COR_CONF, label=f'Confiança ({pct(R.conf)})')
    ax.fill_between(x, y, where=~centro, interpolate=True, color=COR_ERRO, alpha=0.55, label=f'Rejeição (α = {fmt(R.alpha, 3)})')
    ax.plot(x, y, color=COR_TINTA, lw=2)
    for v in (-z, z):
        ax.axvline(v, color='gray', ls='--', lw=1.2)
        ax.text(v, 0.415, sgn(v, 3), ha='center', fontweight='bold')
    ax.text(0, 0.16, pct(R.conf), ha='center', fontsize=18, fontweight='bold')
    ax.set(xlabel='z', xlim=(-lim, lim), ylim=(0, 0.45), title=f'De onde sai o z: z ≈ {fmt(z, 3)}')
    ax.legend(loc='upper left', frameon=False, fontsize=9)
    return ax

g_normal(R); plt.show()

**Margem:** cai com √n — para dividir o erro por 2 é preciso quadruplicar a amostra.

In [ ]:
def g_margem(R, ax=None):
    ax = ax or plt.subplots()[1]
    xmax = max(1e4, 10 ** np.ceil(np.log10(R.n * 3)))
    ns = np.logspace(1, np.log10(xmax), 300)
    m = R.z * np.sqrt(0.25 / ns) * 100
    ax.semilogx(ns, m, color=COR_A, lw=2.6, label='Margem de erro (p = 0,5)')
    ax.fill_between(ns, m, color=COR_A, alpha=0.12)
    ax.axhline(R.e * 100, color='gray', ls='--', label=f'ε planejado = ±{fmt(R.e * 100, 1)} p.p.')
    mn = R.z * np.sqrt(0.25 / R.n) * 100
    ax.plot(R.n, mn, 'o', color=COR_ERRO, ms=10, mec='white', mew=2, label=f'n = {fmt(R.n)} → ±{fmt(mn, 2)} p.p.')
    ax.set(xlabel='Tamanho da amostra (escala log)', ylabel='Margem (p.p.)', ylim=(0, None), title='A margem cai com √n')
    ax.legend(frameon=False, fontsize=9)
    return ax

g_margem(R); plt.show()

tabela = pd.DataFrame({'n': [10, 100, 500, 1000, R.n, 5000]}).drop_duplicates().sort_values('n')
tabela['margem (p.p.)'] = (R.z * np.sqrt(0.25 / tabela['n']) * 100).round(1)
display(tabela.reset_index(drop=True))

## 4. Pesquisa realizada

In [ ]:
def g_resultado(R, ax=None):
    ax = ax or plt.subplots(figsize=(7, 4.8))[1]
    nomes = [NOME_A, NOME_B]
    vals = np.array([1 - R.p, R.p]) * 100
    m = R.moe * 100
    ax.bar(nomes, vals, color=[COR_A, COR_B], width=0.55)
    ax.errorbar(nomes, vals, yerr=m, fmt='none', ecolor=COR_TINTA, elinewidth=2, capsize=16)
    for i, v in enumerate(vals):
        ax.text(i, v + m + 1.5, f'{fmt(v, 1)}% ± {fmt(m, 1)}', ha='center', fontweight='bold')
    ax.axhline(50, color=COR_TINTA, ls='--', lw=1.2)
    ax.set(ylabel='% das respostas', ylim=(0, min(100, vals.max() + m + 10)),
           title=f'n = {fmt(R.n)} entrevistas — {R.veredito}')
    return ax

g_resultado(R); plt.show()

## 5. Da pessoa à contagem: Bernoulli → Binomial

Cada resposta é 0 (Praia) ou 1 (Montanha). Somando n respostas independentes, a contagem K segue uma Binomial — e para n grande ela é praticamente Normal.

In [ ]:
def g_binomial(R, ax=None):
    ax = ax or plt.subplots()[1]
    n, p = R.n, R.p
    mu, sd = n * p, np.sqrt(n * p * (1 - p))
    a, b = int(max(0, np.floor(mu - 4.2 * sd))), int(min(n, np.ceil(mu + 4.2 * sd)))
    mostra_empate = abs(n / 2 - mu) < 6 * sd
    if mostra_empate:
        a, b = min(a, int(n / 2 - sd)), max(b, int(n / 2 + sd))
    k = np.arange(a, b + 1)
    ax.bar(k, stats.binom.pmf(k, n, p), width=0.8, color=COR_B, alpha=0.55, label='P(K = k) exata')
    ax.bar([R.kB], [stats.binom.pmf(R.kB, n, p)], width=0.8, color=COR_B, label=f'observado: k = {fmt(R.kB)}')
    xx = np.linspace(a, b, 400)
    ax.plot(xx, stats.norm.pdf(xx, mu, sd), color=COR_TINTA, lw=2, label='aproximação Normal')
    if mostra_empate:
        ax.axvline(n / 2, color=COR_A, ls='--', lw=1.8, label=f'empate: {fmt(n / 2, 1 if n % 2 else 0)}')
    ax.set(xlabel=f'k = votos em {NOME_B}', ylabel='Probabilidade',
           title=f'K ~ Binomial({fmt(n)}; {fmt(p, 3)})  →  {fmt(mu)} ± {fmt(sd, 1)} votos')
    ax.legend(frameon=False, fontsize=9)
    return ax

g_binomial(R); plt.show()

## 6. Se a pesquisa fosse repetida 10.000 vezes

Simulação de Monte Carlo com `numpy.random`: cada pesquisa gera um p̂ diferente, mas eles se distribuem como uma Normal centrada na proporção real, com desvio igual ao erro padrão.

In [ ]:
def g_simulacao(R, M=10_000, semente=42):
    rng = np.random.default_rng(semente)
    p_sim = rng.binomial(R.n, R.p, size=M) / R.n
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))

    lo, hi = R.p - 4.5 * R.ep, R.p + 4.5 * R.ep
    ax1.hist(p_sim, bins=40, range=(lo, hi), density=True, color=COR_CONF, edgecolor='white', label='pesquisas simuladas')
    xx = np.linspace(lo, hi, 400)
    ax1.plot(xx, stats.norm.pdf(xx, R.p, R.ep), color=COR_TINTA, lw=2, label='Normal(p, EP)')
    ax1.axvline(R.p, color=COR_ERRO, lw=2, label='proporção real')
    for v in (R.p - R.z * R.ep, R.p + R.z * R.ep):
        ax1.axvline(v, color='gray', ls='--')
    ax1.set(xlabel=f'p̂ ({NOME_B})', ylabel='Densidade', title=f'{fmt(M)} pesquisas com n = {fmt(R.n)}')
    ax1.legend(frameon=False, fontsize=9)

    m = np.arange(1, M + 1)
    media = np.cumsum(p_sim) / m
    var = (np.cumsum(p_sim**2) - m * media**2) / np.maximum(m - 1, 1)
    dp = np.sqrt(np.clip(var, 0, None))
    ax2.semilogx(m[9:], dp[9:], color=COR_A, lw=2, label='DP(p̂) observado')
    ax2.axhline(R.ep, color=COR_ERRO, ls='--', lw=1.6, label=f'EP teórico = {fmt(R.ep, 4)}')
    ax2.set(xlabel='Número de repetições', ylabel='DP(p̂)', title='O erro padrão observado converge')
    ax2.legend(frameon=False)

    cobertura = np.mean(np.abs(p_sim - R.p) <= R.z * R.ep)
    fig.suptitle(f'{pct(cobertura)} das pesquisas ficaram a até ±z·EP da proporção real (esperado: {pct(R.conf)})')
    fig.tight_layout()

g_simulacao(R); plt.show()

## 7. Verossimilhança: a pergunta invertida

Com os dados fixos (n e k), qual valor de p os torna mais plausíveis? O máximo fica exatamente em p̂ = k/n.

In [ ]:
def rel_lik(p, n, p_hat):
    '''Verossimilhança relativa L(p)/L(p̂) para k = n·p̂ votos em Montanha.'''
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return np.exp(n * (p_hat * np.log(p / p_hat) + (1 - p_hat) * np.log((1 - p) / (1 - p_hat))))

def faixa(n, p_hat, mult):
    ep = np.sqrt(p_hat * (1 - p_hat) / n)
    return max(0.0005, p_hat - mult * ep), min(0.9995, p_hat + mult * ep)

def g_verossimilhanca(R):
    lo, hi = faixa(R.n, R.p, 5)
    p = np.linspace(lo, hi, 500)
    L = rel_lik(p, R.n, R.p)
    corte = np.exp(-R.z**2 / 2)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
    ax1.fill_between(p, L, color=COR_A, alpha=0.15)
    ax1.plot(p, L, color=COR_A, lw=2.5)
    ax1.axhline(corte, color=COR_OK, ls='--', label=f'corte do IC ≈ {fmt(corte, 3)}')
    ax1.axvline(R.p, color=COR_ERRO, lw=2, label=f'p̂ = {fmt(R.p, 3)}')
    ax2.plot(p, np.log(np.maximum(L, 1e-300)), color=COR_A, lw=2.5)
    ax2.axhline(-R.z**2 / 2, color=COR_OK, ls='--')
    ax2.axvline(R.p, color=COR_ERRO, lw=2)
    if lo < 0.5 < hi:
        l50 = float(rel_lik(0.5, R.n, R.p))
        for ax in (ax1, ax2):
            ax.axvline(0.5, color='gray', ls=':', lw=1.8)
        ax1.plot(0.5, l50, 'o', color='gray')
        ax1.annotate(f'50%: {fmt(l50, 2)}', (0.5, l50), xytext=(6, 8), textcoords='offset points', fontweight='bold')
    ax1.set(xlabel='p', ylabel='L(p) / L(p̂)', ylim=(0, 1.08), title='Verossimilhança relativa')
    ax1.legend(frameon=False, fontsize=9)
    ax2.set(xlabel='p', ylabel='ℓ(p) − ℓ(p̂)', ylim=(-14, 1), title='Log-verossimilhança: parábola invertida')
    fig.tight_layout()

def tabela_candidatos(R):
    passo = max(0.005, round(R.ep * 0.65 * 200) / 200)
    cands = [R.p + j * passo for j in range(-3, 4)]
    if all(abs(c - 0.5) > 1e-9 for c in cands) and abs(0.5 - R.p) < 8 * passo:
        cands.append(0.5)
    corte = np.exp(-R.z**2 / 2)
    linhas = []
    for c in sorted(c for c in cands if 0 < c < 1):
        v = float(rel_lik(c, R.n, R.p))
        if abs(c - R.p) < 1e-9: leitura = 'máximo: o melhor palpite'
        elif v >= 0.5: leitura = 'quase tão bom quanto o melhor'
        elif v >= corte: leitura = 'possível (dentro do IC)'
        elif v >= 0.01: leitura = 'pouco plausível'
        else: leitura = 'implausível'
        rotulo = pct(c) + (' (empate)' if abs(c - 0.5) < 1e-9 else '')
        linhas.append({f'p candidato ({NOME_B})': rotulo, 'L(p)/L(p̂)': fmt(v, 2), 'Leitura': leitura})
    return pd.DataFrame(linhas)

g_verossimilhanca(R); plt.show()
display(tabela_candidatos(R))

## 8. A curvatura devolve o erro padrão

Quanto maior n, mais fechada a curva. A curvatura no topo é a informação observada I(p̂) = n / (p̂(1−p̂)), e EP = 1/√I.

In [ ]:
def g_curvatura(R):
    ns = sorted({v for v in (round(R.n / 10), round(R.n / 4), R.n, R.n * 4, R.n * 16) if v >= 10})
    lo, hi = faixa(ns[0], R.p, 3.4)
    p = np.linspace(lo, hi, 600)
    cores = plt.cm.viridis(np.linspace(0.1, 0.85, len(ns)))
    fig, ax = plt.subplots()
    for v, c in zip(ns, cores):
        atual = v == R.n
        ax.plot(p, rel_lik(p, v, R.p), color=COR_ERRO if atual else c, lw=3.2 if atual else 2,
                label=f'n = {fmt(v)}' + (' (atual)' if atual else ''))
    if lo < 0.5 < hi:
        ax.axvline(0.5, color='gray', ls=':', lw=1.8, label='empate (50%)')
    ax.set(xlabel='p', ylabel='L(p) / L(p̂)', ylim=(0, 1.08), title='Mesmo p̂, amostras de tamanhos diferentes')
    ax.legend(frameon=False, fontsize=9)
    plt.show()
    info = np.array(ns) / R.s2
    display(pd.DataFrame({'n': [fmt(v) for v in ns],
                          'Informação I(p̂)': [fmt(i) for i in info],
                          'Erro padrão': [fmt(100 / np.sqrt(i), 2) + ' p.p.' for i in info]}))

g_curvatura(R)

## 9. Intervalo de confiança: o jeito certo de comparar

Não basta ver se as barras se sobrepõem: olhamos o intervalo da **diferença** D = Montanha − Praia. Se ele contém zero, é empate técnico.

In [ ]:
def g_ic(R):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8))
    m, e = R.moe * 100, R.e * 100
    for y, v, cor in [(1, (1 - R.p) * 100, COR_A), (0, R.p * 100, COR_B)]:
        ax1.barh(y, 2 * e, left=v - e, height=0.5, color=COR_A, alpha=0.12, edgecolor=COR_A)
        ax1.plot([v - m, v + m], [y, y], color=COR_TINTA, lw=6, solid_capstyle='round')
        ax1.plot(v, y, 'o', color=cor, ms=14, mec='white', mew=3)
        ax1.text(v - m, y + 0.32, f'{fmt(v - m, 1)}%', ha='center', fontsize=9)
        ax1.text(v + m, y + 0.32, f'{fmt(v + m, 1)}%', ha='center', fontsize=9)
    ax1.axvline(50, color=COR_TINTA, ls='--')
    ax1.set(yticks=[0, 1], yticklabels=[NOME_B, NOME_A], ylim=(-0.6, 1.7), xlabel='%',
            title=f'IC de {pct(R.conf)} para cada destino')

    a, b = R.ic_D[0] * 100, R.ic_D[1] * 100
    ax2.plot([a, b], [0, 0], color=COR_ERRO if R.empate else COR_OK, lw=9, solid_capstyle='round')
    ax2.plot(R.D * 100, 0, 'o', color=COR_TINTA, ms=14, mec='white', mew=3)
    ax2.axvline(0, color=COR_TINTA, ls='--')
    span = max(abs(a), abs(b)) * 1.25 + 0.5
    ax2.set(xlim=(-span, span), ylim=(-1, 1), yticks=[], xlabel='p.p.',
            title=f'D = {NOME_B} − {NOME_A}: [{sgn(a)}; {sgn(b)}] p.p. → {R.veredito}')
    fig.tight_layout()

g_ic(R); plt.show()
print(f'Para desempatar com 80% de poder, mantendo a diferença observada: ≈ {fmt(R.n_poder)} entrevistas.')

## 10. Painel interativo

Mexa nos controles: todos os números e gráficos são recalculados. Em **n**, deixe 0 para usar a amostra mínima calculada.

In [ ]:
def painel(conf=95.0, eps=3.0, pB=51.0, N=15_000_000, n=0):
    R = calcular(N=N, conf=conf, eps=eps, pB=pB, n=n)
    resumo(R)
    fig, axs = plt.subplots(2, 2, figsize=(14, 9))
    g_normal(R, axs[0, 0])
    g_margem(R, axs[0, 1])
    g_resultado(R, axs[1, 0])
    g_binomial(R, axs[1, 1])
    fig.tight_layout(); plt.show()
    g_ic(R); plt.show()

estilo = {'description_width': '110px'}
widgets.interact(
    painel,
    conf=widgets.FloatSlider(value=95, min=80, max=99.9, step=0.1, description='Confiança (%)', style=estilo, continuous_update=False),
    eps=widgets.FloatSlider(value=3, min=0.5, max=10, step=0.1, description='Margem (p.p.)', style=estilo, continuous_update=False),
    pB=widgets.FloatSlider(value=51, min=1, max=99, step=0.5, description=f'% {NOME_B}', style=estilo, continuous_update=False),
    N=widgets.Dropdown(options=[('10 mil', 10_000), ('100 mil', 100_000), ('1 milhão', 1_000_000),
                                ('15 milhões', 15_000_000), ('210 milhões', 210_000_000)],
                       value=15_000_000, description='População N', style=estilo),
    n=widgets.IntText(value=0, description='n (0 = mínimo)', style=estilo),
);